# Excepciones y Depuración

**Objetivos:**
- Entender excepciones en Python y su manejo con `try/except/else/finally`.
- Usar `raise` y `assert` correctamente (nota: `assert` puede ignorarse en producción).
- Conocer técnicas de depuración (print, logging, depurador, pruebas) y usar LLMs como ayuda.

## 1) ¿Qué es una excepción?
- Una excepción es un evento que ocurre durante la ejecución y rompe el flujo normal.
- Python lanza objetos de excepción (tipo, mensaje y rastreo).
- Ejemplos comunes: `ZeroDivisionError`, `FileNotFoundError`, `TypeError`, `KeyError`.

**Idea:** usar excepciones para manejar casos inesperados sin que el programa se caiga.

In [1]:
# Ejemplo sencillo: captura de división por cero
def dividir(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print('Error: División por cero detectada')
        return None

print(dividir(10, 2))  # 5.0
print(dividir(5, 0))   # maneja ZeroDivisionError

5.0
Error: División por cero detectada
None


## 2) `try` / `except` / `else` / `finally`
- `try`: código que puede fallar.
- `except`: maneja excepciones específicas (evitar `except:` genérico).
- `else`: se ejecuta si no hubo excepción.
- `finally`: siempre se ejecuta (ideal para cerrar recursos).

In [2]:
# Ejemplo con archivo (simulado)
def procesar_archivo(path):
    try:
        f = open(path, 'r', encoding='utf-8')
    except FileNotFoundError as e:
        print('No se encontró el archivo:', e)
        return None
    else:
        data = f.read()
        print('Leídos', len(data), 'caracteres')
        return data
    finally:
        try:
            f.close()
            print('Archivo cerrado (si fue abierto)')
        except Exception:
            pass

# Prueba (archivo probablemente inexistente en este entorno)
procesar_archivo('no_existe.txt')

No se encontró el archivo: [Errno 2] No such file or directory: 'no_existe.txt'


In [3]:
procesar_archivo('01_Excepciones.md')

Leídos 4749 caracteres
Archivo cerrado (si fue abierto)


'# Resumen sobre Excepciones en Python\n\n## Introducción\n- Se compara la construcción de un rascacielos con la gestión de imprevistos en programación.\n- Las excepciones en Python son alertas que indican problemas durante la ejecución del código.\n\n## Definición de Excepciones\n- Una excepción es como una luz de control del motor, avisando sobre problemas antes de que causen fallos.\n- Interrumpen el flujo normal del programa y permiten resolver problemas antes de que el código se bloquee.\n\n## Tipos de Excepciones\n- Ejemplos comunes: división por cero, archivo no encontrado, entrada de usuario no válida.\n- Al ocurrir una excepción, se lanza un objeto de excepción que contiene:\n  - Tipo de excepción\n  - Rastreo de la ejecución\n  - Mensaje de error\n\n## Manejo de Excepciones\n- Las excepciones se gestionan con bloques `try` y `except`.\n  - **Bloque `try`**: Contiene el código que puede generar una excepción.\n  - **Bloque `except`**: Especifica cómo manejar la excepción si oc

In [ ]:
def procesar_archivo(path):
    try:
        f = open(path, 'r', encoding='utf-8')
    except FileNotFoundError as e:
        print('No se encontró el archivo:', e)
        return None
    except Exception as e:
        print('Ocurrió un error inesperado:', e)
        return None
    else:
        data = f.read()
        print('Leídos', len(data), 'caracteres')
        return data
    finally:
        try:
            f.close()
            print('Archivo cerrado (si fue abierto)')
        except Exception:
            pass


## 3) `raise` y excepciones personalizadas
- `raise` permite lanzar una excepción intencionadamente cuando detectas una condición inválida.
- Puedes definir clases de excepción propias heredando de `Exception` para mayor claridad.


In [4]:
# Ejemplo de raise y excepción personalizada
class DatosInvalidosError(Exception):
    pass

def validar_edad(edad):
    if edad < 0 or edad > 120:
        raise DatosInvalidosError(f'Edad inválida: {edad}')
    return True

try:
    validar_edad(-5)
except DatosInvalidosError as e:
    print('Capturada excepción personalizada:', e)
except Exception as e:
    print('Ocurrió un error inesperado:', e)


Capturada excepción personalizada: Edad inválida: -5


## 4) `assert` (uso y precauciones)
- `assert condición, 'mensaje'` se usa para comprobar invariantes durante el desarrollo.
- Atención: si Python se ejecuta con optimizaciones (`-O`), las aserciones se ignoran; no uses `assert` para lógica crítica en producción.


In [5]:
# Ejemplo de assert
def promedio(lista):
    assert len(lista) > 0, 'La lista no puede estar vacía'
    return sum(lista) / len(lista)

print(promedio([1,2,3]))
print(promedio([]))  # lanzaría AssertionError (en modo normal)


2.0


AssertionError: La lista no puede estar vacía

## 5) Buenas prácticas con excepciones
- Capturar excepciones específicas en lugar de atrapar todo (`except Exception:`).
- Registrar (log) errores con suficiente contexto para reproducirlos.
- No usar excepciones para controlar flujo normal; reservarlas para casos excepcionales.
- Normalizar mensajes y, cuando sea apropiado, re-lanzar excepciones con información adicional.


## 6) Métodos de depuración
- Depuración por impresión (`print`) — rápido y directo para scripts pequeños.
- `logging` — registra con niveles (DEBUG/INFO/WARNING/ERROR) para producción.
- Depurador interactivo (`pdb` / depurador del IDE) — inspección paso a paso y breakpoints.
- Aserciones para detectar invariantes durante desarrollo.
- Pruebas unitarias para prevenir regresiones.


In [6]:
# Ejemplo de logging en vez de prints
import logging
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s:%(message)s')

def proceso(x):
    logging.debug('Entrando en proceso con x=%s', x)
    if x < 0:
        logging.warning('x es negativo: %s', x)
    return x * 2

print(proceso(3))
print(proceso(-1))


DEBUG:Entrando en proceso con x=3
DEBUG:Entrando en proceso con x=-1


6
-2


In [9]:
# En un notebook, esto activa el modo de depuración para la última excepción
# Ejemplo de uso de pdb en un script (en notebooks set_trace puede abrir interfaz diferente)
# En un script:
# import pdb; pdb.set_trace()
# En un notebook puedes usar %debug o herramientas del IDE.

def f(a, b):
    c = a + b
    return c

print(f(2,3))


5


## 7) Uso de LLMs (p. ej. ChatGPT) para depuración — cómo y cuándo
- Para explicar trazas de error: pega el *traceback* y pregunta por causas probables.
- Para proponer correcciones, fragmentos de prueba o pruebas unitarias.
- Para generar ejemplos y explicación en lenguaje simple para estudiantes.
- Buen prompt ejemplo:

Precauciones: no compartas secretos (credenciales, datos sensibles) y revisa siempre la solución propuesta antes de aplicarla.
Precauciones: no compartas secretos (credenciales, datos sensibles) y revisa siempre la solución propuesta antes de aplicarla.

## 7.b) Ejemplos de excepciones comunes en Python
A continuación hay ejemplos cortos que muestran cómo se generan y manejan excepciones habituales. Útiles para explicar en clase y relacionar con las preguntas del examen.


In [ ]:
# Ejemplos de excepciones comunes
# NameError
try:
    print(x)
except NameError as e:
    print('NameError capturada:', e)

# TypeError
try:
    1 + 'a'
except TypeError as e:
    print('TypeError capturada:', e)

# IndexError
try:
    lst = [1,2]
    print(lst[5])
except IndexError as e:
    print('IndexError capturada:', e)

# KeyError
try:
    d = {'a':1}
    print(d['b'])
except KeyError as e:
    print('KeyError capturada:', e)

# ValueError
try:
    int('no_num')
except ValueError as e:
    print('ValueError capturada:', e)

# FileNotFoundError
try:
    open('archivo_que_no_existe.txt')
except FileNotFoundError as e:
    print('FileNotFoundError capturada:', e)

# ZeroDivisionError
try:
    1/0
except ZeroDivisionError as e:
    print('ZeroDivisionError capturada:', e)

# AttributeError / ImportError (ejemplos rápidos)
try:
    'a'.no_exist()
except AttributeError as e:
    print('AttributeError capturada:', e)

try:
    import modulo_no_existente
except ImportError as e:
    print('ImportError capturada:', e)


## 7.c) Depuradores del IDE (importante para el examen)
- Objetivo: cubrir las funcionalidades que los alumnos deben conocer para el examen y la práctica en clase.
- Funciones clave a mostrar y practicar en el IDE (ej. VS Code o PyCharm):
  - Establecer puntos de interrupción (breakpoints).
  - Ejecutar hasta el breakpoint y continuar (`Continue`).
  - `Step Over` (pasar por encima), `Step Into` (entrar), `Step Out` (salir).
  - Inspeccionar variables locales y ver/editar valores en tiempo de ejecución (Variables/Watch).
  - Ver la pila de llamadas (Call Stack) y navegar entre marcos (frames).
  - Evaluar expresiones en la consola de depuración (Debug Console).
  - Añadir condiciones a breakpoints (breakpoint condicional) y excepciones que detengan la ejecución.

- Demostración rápida (pasos en VS Code):
  1) Abra el archivo `ej.py` o el script de ejemplo en VS Code.
  2) Haga clic en la barra izquierda para añadir un breakpoint en la línea deseada.
  3) Inicie 'Run and Debug' → seleccione 'Python File'.
  4) Use `Step Into`/`Step Over` y examine variables en el panel 'Variables'.

- Relación con preguntas del examen: asegúrate de practicar que los alumnos sepan cómo inspeccionar variables, usar breakpoints y ejecutar paso a paso — estas habilidades aparecen en [11_ Examen.md](Fundamentos de programacion en Python/05_Execpciones_Debug/11_%20Examen.md).

## 8) Ejercicios rápidos (10–15 min)
1. Escribe una función que lea un número desde un archivo y maneje `FileNotFoundError` y `ValueError`.
2. Crea una excepción personalizada `FormatoIncorrectoError` y lánzala si el contenido no cumple un formato simple.
3. Añade `logging` y una prueba unitaria sencilla usando `pytest` o `unittest`.

Sugerencia de tiempo en clase: 5 min explicación + 10 min práctica + 10 min revisión/feedback.

## 9) Resumen y recursos
- `try/except/else/finally`, `raise`, `assert` y buenas prácticas: clave para código robusto.
- Depuración: combinar prints, logging, depurador y pruebas unitarias.
- LLMs: gran ayuda pedagógica y para sugerir soluciones, siempre validar resultados.

Recursos: documentación oficial de Python (`docs.python.org`), tutoriales sobre `logging` y `pdb`.

---
Hecho: Notebook creado en esta carpeta; dime si quieres que lo ajuste (más ejercicios, traducciones o añadir outputs).